# RAG Demo (Local Ollama + Chroma) — with Artifact Caching (Jupyter)

This notebook is an end-to-end **local** RAG demo:

- Local LLM + embeddings via **Ollama**
- Vector DB via **ChromaDB** (persistent on disk)
- Word-based chunking with overlap
- Retrieval → context assembly → grounded answer with citations
- **Quality-of-life improvements**:
  - Downloads cached in `CORPUS_DIR`
  - Artifacts cached in `ARTIFACTS_DIR` (`chunks.json`, `embeddings.npy`, `manifest.json`)
  - On rerun: loads precomputed data and **skips recomputation**
  - If Chroma DB is missing but artifacts exist: rebuilds Chroma from artifacts without re-embedding

You need:
- Ollama installed and running
- A chat model and an embedding model pulled locally (`ollama list`)


## 0) Install dependencies
Run once per environment.

In [ ]:
# If needed, uncomment:
# !pip install -q chromadb requests tqdm numpy


import os, re, json, time, hashlib
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Any, Optional, Tuple

import numpy as np
import requests
import chromadb

try:
    from tqdm import tqdm
except Exception:
    tqdm = None  # optional progress bars


import textwrap



def pretty_print(*args):
    text = " ".join(str(arg) for arg in args)
    try:
        print(textwrap.fill(text, width=80))
    except Exception as e:
        print(text)  # fallback to normal print if text is not a string




## 1) Configuration
Edit these values if you want.

In [13]:
# === Configuration (change these as needed) ===
@dataclass
class Settings:
    # Ollama
    OLLAMA_URL: str = os.getenv("OLLAMA_URL", "http://localhost:11434")
    CHAT_MODEL: str = os.getenv("CHAT_MODEL", "llama3.1:8b")
    EMBED_MODEL: str = os.getenv("EMBED_MODEL", "embeddinggemma:latest")   # e.g. "nomic-embed-text"

    # Chunking
    WORDS_PER_CHUNK: int = int(os.getenv("WORDS_PER_CHUNK", "300"))
    OVERLAP_WORDS: int = int(os.getenv("OVERLAP_WORDS", "60"))

    # Retrieval
    TOPK: int = int(os.getenv("TOPK", "5"))

    # Storage
    CORPUS_DIR: str = os.getenv("CORPUS_DIR", "corpus_jupyter")
    ARTIFACTS_DIR: str = os.getenv("ARTIFACTS_DIR", "rag_artifacts")
    CHROMA_PATH: str = os.getenv("CHROMA_PATH", "./chroma_db")
    CHROMA_COLLECTION: str = os.getenv("CHROMA_COLLECTION", "rag_demo")
    CHROMA_DISTANCE: str = os.getenv("CHROMA_DISTANCE", "cosine")  # cosine is a good default

    # Workflow toggles
    DOWNLOAD_FROM_WEB: bool = (os.getenv("DOWNLOAD_FROM_WEB", "true").lower() == "true")
    FORCE_REBUILD: bool = (os.getenv("FORCE_REBUILD", "false").lower() == "true")

    # Performance
    EMBED_BATCH_SIZE: int = int(os.getenv("EMBED_BATCH_SIZE", "64"))  # batched /api/embed

SETTINGS = Settings()
SETTINGS


Settings(OLLAMA_URL='http://localhost:11434', CHAT_MODEL='llama3.1:8b', EMBED_MODEL='embeddinggemma:latest', WORDS_PER_CHUNK=300, OVERLAP_WORDS=60, TOPK=5, CORPUS_DIR='corpus_jupyter', ARTIFACTS_DIR='rag_artifacts', CHROMA_PATH='./chroma_db', CHROMA_COLLECTION='rag_demo', CHROMA_DISTANCE='cosine', DOWNLOAD_FROM_WEB=True, FORCE_REBUILD=False, EMBED_BATCH_SIZE=64)

## 2) Corpus list (Project Gutenberg)
Disable downloads if you want to stay offline (then put files in `CORPUS_DIR`).

In [14]:
GUTENBERG_BOOKS = {
    "Moby-Dick": "https://www.gutenberg.org/files/2701/2701-0.txt",
    "Pride and Prejudice": "https://www.gutenberg.org/files/1342/1342-0.txt",
    "Frankenstein": "https://www.gutenberg.org/files/84/84-0.txt",
    "Alice in Wonderland": "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "Dracula": "https://www.gutenberg.org/files/345/345-0.txt",
    "A Tale of Two Cities": "https://www.gutenberg.org/files/98/98-0.txt",
    "The Great Gatsby": "https://www.gutenberg.org/cache/epub/64317/pg64317.txt",
    "Adventures of Sherlock Holmes": "https://www.gutenberg.org/files/1661/1661-0.txt",
    "War and Peace": "https://www.gutenberg.org/files/2600/2600-0.txt",
    "Jane Eyre": "https://www.gutenberg.org/files/1260/1260-0.txt",
    "The Picture of Dorian Gray": "https://www.gutenberg.org/files/174/174-0.txt",
    "Crime and Punishment": "https://www.gutenberg.org/files/2554/2554-0.txt",
    "Wuthering Heights": "https://www.gutenberg.org/files/768/768-0.txt",
}

GUTENBERG = [(t, u) for t, u in GUTENBERG_BOOKS.items()]
len(GUTENBERG)


13

## 3) Ollama helpers
Uses the modern batched embed endpoint when available, with a safe fallback.

In [15]:

# Create a session that ignores proxy env vars (bypasses VPN for local Ollama)
SESSION = requests.Session()
SESSION.trust_env = False  # equivalent of ollama.Client(..., trust_env=False)


def _url(path):
    return SETTINGS.OLLAMA_URL.rstrip('/') + path

def ollama_list_models():
    r = SESSION.get(_url("/api/tags"), timeout=30)
    r.raise_for_status()
    data = r.json()
    return [m.get("name","") for m in (data.get("models") or [])]

def check_ollama_ready():
    models = ollama_list_models()
    print(f"Ollama reachable. {len(models)} model(s) found.")
    if SETTINGS.CHAT_MODEL not in models:
        print(f"Warning: chat model '{SETTINGS.CHAT_MODEL}' not found in `ollama list`.")
    if SETTINGS.EMBED_MODEL not in models:
        print(f"Warning: embed model '{SETTINGS.EMBED_MODEL}' not found in `ollama list`.")

def ollama_embed(texts):
    """Embed many texts. Prefer /api/embed (batched). Fallback to /api/embeddings."""
    if not texts:
        return []
    # Prefer modern batched API
    try:
        r = SESSION.post(
            _url("/api/embed"),
            json={"model": SETTINGS.EMBED_MODEL, "input": texts},
            timeout=600,
        )
        r.raise_for_status()
        data = r.json()
        embs = data.get("embeddings")
        if not embs:
            raise RuntimeError(f"Missing 'embeddings' in /api/embed response: {data}")
        return embs
    except Exception:
        # Fallback: older per-text endpoint
        out = []
        for t in texts:
            r = SESSION.post(
                _url("/api/embeddings"),
                json={"model": SETTINGS.EMBED_MODEL, "prompt": t},
                timeout=600,
            )
            r.raise_for_status()
            data = r.json()
            emb = data.get("embedding")
            if not emb:
                raise RuntimeError(f"Missing 'embedding' in /api/embeddings response: {data}")
            out.append(emb)
        return out

def ollama_chat(messages, temperature=0.2):
    r = SESSION.post(
        _url("/api/chat"),
        json={"model": SETTINGS.CHAT_MODEL, "messages": messages, "stream": False, "options": {"temperature": temperature}},
        timeout=600,
    )
    r.raise_for_status()
    data = r.json()
    return ((data.get("message") or {}).get("content")) or data.get("response","") or ""

check_ollama_ready()


Ollama reachable. 9 model(s) found.


## 4) Text + chunking utilities

In [ ]:
START_MARK = re.compile(r"\*\*\* START OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)
END_MARK   = re.compile(r"\*\*\* END OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)

def strip_gutenberg_boilerplate(txt):
    start = START_MARK.search(txt)
    end = END_MARK.search(txt)
    if start and end and end.start() > start.end():
        return txt[start.end():end.start()].strip()
    return txt.strip()

def word_chunks(text, max_words, overlap):
    text = re.sub(r"\s+", " ", text).strip()
    words = text.split()
    if not words:
        return []
    step = max(1, max_words - overlap)
    chunks = []
    for start in range(0, len(words), step):
        end = start + max_words
        seg = words[start:end]
        if len(seg) < max(60, max_words // 4):
            break
        chunks.append(" ".join(seg))
        if end >= len(words):
            break
    return chunks

def l2_normalize(mat):
    norms = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12
    return mat / norms


## 5) Artifact + Chroma helpers
This is the part that prevents recomputation.

In [6]:
def ensure_dir(p):
    path = Path(p)
    path.mkdir(parents=True, exist_ok=True)
    return path

def artifacts_paths():
    base = ensure_dir(SETTINGS.ARTIFACTS_DIR)
    return {
        "base": base,
        "manifest": base / "manifest.json",
        "chunks": base / "chunks.json",
        "embeddings": base / "embeddings.npy",
    }

def corpus_file_path(title):
    safe = re.sub(r"[^a-zA-Z0-9_-]+", "_", title).strip("_")
    return ensure_dir(SETTINGS.CORPUS_DIR) / f"{safe}.txt"

def file_fingerprint(path):
    st = path.stat()
    return f"{path.name}|{st.st_size}|{int(st.st_mtime)}"

def corpus_fingerprint(doc_paths):
    s = "\n".join([file_fingerprint(p) for p in doc_paths if p.exists()])
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def load_manifest():
    p = artifacts_paths()["manifest"]
    if not p.exists():
        return None
    return json.loads(p.read_text(encoding="utf-8"))

def save_manifest(manifest):
    p = artifacts_paths()["manifest"]
    p.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

def chroma_client():
    return chromadb.PersistentClient(path=SETTINGS.CHROMA_PATH)

def get_or_create_collection(recreate=False):
    client = chroma_client()
    if recreate:
        try:
            client.delete_collection(SETTINGS.CHROMA_COLLECTION)
        except Exception:
            pass
    return client.get_or_create_collection(
        name=SETTINGS.CHROMA_COLLECTION,
        metadata={"hnsw:space": SETTINGS.CHROMA_DISTANCE},
    )

def chroma_count():
    return get_or_create_collection().count()

def settings_signature():
    return {
        "OLLAMA_URL": SETTINGS.OLLAMA_URL,
        "CHAT_MODEL": SETTINGS.CHAT_MODEL,
        "EMBED_MODEL": SETTINGS.EMBED_MODEL,
        "WORDS_PER_CHUNK": SETTINGS.WORDS_PER_CHUNK,
        "OVERLAP_WORDS": SETTINGS.OVERLAP_WORDS,
        "CHROMA_COLLECTION": SETTINGS.CHROMA_COLLECTION,
        "CHROMA_DISTANCE": SETTINGS.CHROMA_DISTANCE,
    }

def manifest_matches_current(manifest, doc_paths):
    if not manifest:
        return False
    if manifest.get("settings") != settings_signature():
        return False
    expected_fp = manifest.get("corpus_fingerprint")
    if not expected_fp:
        return False
    return expected_fp == corpus_fingerprint(doc_paths)

def try_load_cached_state():
    ap = artifacts_paths()
    ensure_dir(SETTINGS.CORPUS_DIR)

    doc_paths = [corpus_file_path(title) for title, _ in GUTENBERG]
    manifest = load_manifest()

    state = {
        "has_manifest": bool(manifest),
        "manifest_ok": False,
        "has_chunks": ap["chunks"].exists(),
        "has_embeddings": ap["embeddings"].exists(),
        "chroma_count": chroma_count(),
        "expected_chunks": (manifest or {}).get("chunk_count"),
        "can_skip_all": False,
        "can_restore_chroma_from_artifacts": False,
        "doc_paths": doc_paths,
        "manifest": manifest,
    }

    if manifest and manifest_matches_current(manifest, doc_paths):
        state["manifest_ok"] = True

    if state["manifest_ok"] and isinstance(state["expected_chunks"], int) and state["chroma_count"] == state["expected_chunks"]:
        state["can_skip_all"] = True
        return state

    if state["manifest_ok"] and ap["chunks"].exists() and ap["embeddings"].exists():
        state["can_restore_chroma_from_artifacts"] = True

    return state

state = try_load_cached_state()
state


{'has_manifest': True,
 'manifest_ok': True,
 'has_chunks': True,
 'has_embeddings': True,
 'chroma_count': 0,
 'expected_chunks': 8509,
 'can_skip_all': False,
 'can_restore_chroma_from_artifacts': True,
 'doc_paths': [PosixPath('corpus_jupyter/Moby-Dick.txt'),
  PosixPath('corpus_jupyter/Pride_and_Prejudice.txt'),
  PosixPath('corpus_jupyter/Frankenstein.txt'),
  PosixPath('corpus_jupyter/Alice_in_Wonderland.txt'),
  PosixPath('corpus_jupyter/Dracula.txt'),
  PosixPath('corpus_jupyter/A_Tale_of_Two_Cities.txt'),
  PosixPath('corpus_jupyter/The_Great_Gatsby.txt'),
  PosixPath('corpus_jupyter/Adventures_of_Sherlock_Holmes.txt'),
  PosixPath('corpus_jupyter/War_and_Peace.txt'),
  PosixPath('corpus_jupyter/Jane_Eyre.txt'),
  PosixPath('corpus_jupyter/The_Picture_of_Dorian_Gray.txt'),
  PosixPath('corpus_jupyter/Crime_and_Punishment.txt'),
  PosixPath('corpus_jupyter/Wuthering_Heights.txt')],
 'manifest': {'created_at': '2026-03-04T22:16:15',
  'settings': {'OLLAMA_URL': 'http://localhost

## 6) Download/load corpus (cached in CORPUS_DIR)

- If `DOWNLOAD_FROM_WEB=True`: downloads missing files and caches them to disk.
- If `DOWNLOAD_FROM_WEB=False`: expects files to already exist in `CORPUS_DIR`.


In [ ]:
def download_or_load_corpus():
    ensure_dir(SETTINGS.CORPUS_DIR)
    docs = []

    for title, url in GUTENBERG:
        out_path = corpus_file_path(title)

        if not out_path.exists():
            if not SETTINGS.DOWNLOAD_FROM_WEB:
                continue
            print(f"Downloading: {title}")
            r = SESSION.get(url, timeout=180)
            r.raise_for_status()
            clean = strip_gutenberg_boilerplate(r.text)
            out_path.write_text(clean, encoding="utf-8")

        text = out_path.read_text(encoding="utf-8", errors="ignore")
        docs.append({"title": title, "url": url, "path": str(out_path), "text": text})

    if not docs:
        raise RuntimeError(
            "No documents loaded. Either enable DOWNLOAD_FROM_WEB, or place .txt files in CORPUS_DIR."
        )
    return docs


## 7) Build or load artifacts + Chroma

This cell is the main entry point. It will:

1) Skip everything if cached state is valid  
2) Otherwise, either:
   - Restore Chroma from `chunks.json` + `embeddings.npy` (fast), or
   - Download → chunk → embed → store artifacts → add to Chroma (slow)


In [8]:
# === Helper: Convert title to URL-safe ID ===
def slugify(s):
    s = s.strip().lower()
    s = re.sub(r"[^a-z0-9\s-]", "", s)
    s = re.sub(r"[\s_-]+", "-", s).strip("-")
    return s or "doc"


# === Step 1: Split documents into chunks ===
def build_chunks(docs):
    chunks = []
    for d in docs:
        doc_id = slugify(d["title"])
        pieces = word_chunks(d["text"], SETTINGS.WORDS_PER_CHUNK, SETTINGS.OVERLAP_WORDS)
        for i, piece in enumerate(pieces):
            chunks.append({
                "id": f"{doc_id}#{i}",
                "doc_id": doc_id,
                "chunk_index": i,
                "title": d["title"],
                "source": d.get("url", ""),
                "text": piece,
            })
    return chunks


# === Step 2: Generate embeddings for all chunks ===
def embed_all_chunks(chunks):
    bs = max(1, SETTINGS.EMBED_BATCH_SIZE)
    embs = []
    it = range(0, len(chunks), bs)
    if tqdm:
        it = tqdm(it, desc=f"Embedding ({SETTINGS.EMBED_MODEL})", total=(len(chunks)+bs-1)//bs)

    for start in it:
        batch_texts = [c["text"] for c in chunks[start:start+bs]]
        batch_embs = ollama_embed(batch_texts)
        arr = np.asarray(batch_embs, dtype=np.float32)
        embs.append(arr)

    mat = np.vstack(embs) if embs else np.zeros((0, 0), dtype=np.float32)
    mat = l2_normalize(mat)  # good for cosine usage
    return mat


# === Step 3: Save artifacts to disk (for caching) ===
def save_artifacts(chunks, embeddings, docs):
    ap = artifacts_paths()
    ap["chunks"].write_text(json.dumps(chunks, ensure_ascii=False), encoding="utf-8")
    np.save(ap["embeddings"], embeddings.astype(np.float32))

    doc_paths = [Path(d["path"]) for d in docs]
    manifest = {
        "created_at": __import__("datetime").datetime.now().isoformat(timespec="seconds"),
        "settings": settings_signature(),
        "chunk_count": int(len(chunks)),
        "embedding_dim": int(embeddings.shape[1]) if embeddings.size else None,
        "corpus_fingerprint": corpus_fingerprint(doc_paths),
        "docs": [
            {
                "title": d["title"],
                "url": d.get("url",""),
                "path": d["path"],
                "words": len((d.get("text") or "").split()),
                "fingerprint": file_fingerprint(Path(d["path"])),
            }
            for d in docs
        ],
    }
    save_manifest(manifest)


# === Fast path: Restore Chroma from cached artifacts ===
def restore_chroma_from_artifacts():
    ap = artifacts_paths()
    chunks = json.loads(ap["chunks"].read_text(encoding="utf-8"))
    emb = np.load(str(ap["embeddings"]))
    if len(chunks) != emb.shape[0]:
        raise RuntimeError(f"Artifact mismatch: {len(chunks)} chunks but embeddings {emb.shape}")

    coll = get_or_create_collection(recreate=True)
    ids = [c["id"] for c in chunks]
    docs = [c["text"] for c in chunks]
    metas = [{"doc_id": c["doc_id"], "chunk_index": c["chunk_index"], "title": c["title"], "source": c["source"]} for c in chunks]

    bs = 2048
    for start in range(0, len(ids), bs):
        coll.add(
            ids=ids[start:start+bs],
            documents=docs[start:start+bs],
            metadatas=metas[start:start+bs],
            embeddings=emb[start:start+bs].tolist(),
        )


# === Full pipeline: Download → Chunk → Embed → Store ===
def build_everything_from_scratch():
    docs = download_or_load_corpus()
    chunks = build_chunks(docs)
    print(f"Built {len(chunks)} chunks. Embedding next...")
    emb = embed_all_chunks(chunks)
    print(f"Embeddings: {emb.shape}")

    coll = get_or_create_collection(recreate=True)

    ids = [c["id"] for c in chunks]
    docs_txt = [c["text"] for c in chunks]
    metas = [{"doc_id": c["doc_id"], "chunk_index": c["chunk_index"], "title": c["title"], "source": c["source"]} for c in chunks]

    bs = 2048
    for start in range(0, len(ids), bs):
        coll.add(
            ids=ids[start:start+bs],
            documents=docs_txt[start:start+bs],
            metadatas=metas[start:start+bs],
            embeddings=emb[start:start+bs].tolist(),
        )

    save_artifacts(chunks, emb, docs)
    print(f"Saved artifacts to: {SETTINGS.ARTIFACTS_DIR}")


# === Main entry point: Smart caching logic ===
def ensure_ready():
    st = try_load_cached_state()

    if SETTINGS.FORCE_REBUILD:
        print("FORCE_REBUILD=True -> rebuilding everything.")
        build_everything_from_scratch()
        return

    if st["can_skip_all"]:
        print("Cache hit: Chroma + artifacts match current settings/corpus. Skipping rebuild.")
        return

    if st["can_restore_chroma_from_artifacts"]:
        print("Chroma missing/mismatched, but artifacts are valid. Restoring Chroma from artifacts...")
        restore_chroma_from_artifacts()
        print("Restore complete.")
        return

    print("Cache miss: building from scratch (download/chunk/embed/add + save artifacts).")
    build_everything_from_scratch()


ensure_ready()
print("Chroma vectors:", chroma_count())


Chroma missing/mismatched, but artifacts are valid. Restoring Chroma from artifacts...
Restore complete.
Chroma vectors: 8509


## 8) Retrieval + context assembly
We embed the query, retrieve top-k chunks, and build a context block with citations.

In [26]:
def retrieve(query, k=None):
    k = k or SETTINGS.TOPK
    coll = get_or_create_collection()

    q = np.asarray(ollama_embed([query])[0], dtype=np.float32)[None, :]
    q = l2_normalize(q)[0].tolist()

    res = coll.query(
        query_embeddings=[q],
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )

    docs = (res.get("documents") or [[]])[0]
    metas = (res.get("metadatas") or [[]])[0]
    dists = (res.get("distances") or [[]])[0]

    hits = []
    for doc, meta, dist in zip(docs, metas, dists):
        meta = meta or {}
        hits.append({
            "text": doc or "",
            "doc_id": meta.get("doc_id", "unknown"),
            "chunk_index": meta.get("chunk_index", -1),
            "title": meta.get("title", ""),
            "source": meta.get("source", ""),
            "distance": float(dist) if dist is not None else None,
        })
    return hits

def build_context(hits):
    print(f"Retrieved {len(hits)} hits.", hits)
    if not hits:
        return "NO_CONTEXT"
    blocks = []
    for h in hits:
        cite = f"[{h.get('doc_id')}#{h.get('chunk_index')}]"
        header = f"{cite} {h.get('title')}"
        blocks.append(header + "\n" + (h.get("text") or "") + "\n---\n")
    return "".join(blocks)

hits = retrieve("Who is Elizabeth Bennet?")
for hit in hits:
    print(f"{hit['title']} (doc_id={hit['doc_id']} chunk={hit['chunk_index']} distance={hit['distance']:.4f})")
pretty_print(build_context(hits)[:100])

Pride and Prejudice (doc_id=pride-and-prejudice chunk=23 score=0.4757)
Pride and Prejudice (doc_id=pride-and-prejudice chunk=490 score=0.5044)
Pride and Prejudice (doc_id=pride-and-prejudice chunk=482 score=0.5237)
Pride and Prejudice (doc_id=pride-and-prejudice chunk=486 score=0.5250)
Pride and Prejudice (doc_id=pride-and-prejudice chunk=34 score=0.5263)
Retrieved 5 hits. [{'text': 'his marrying whichever he chooses of the girls--though I must throw in a good word for my little Lizzy.” “I desire you will do no such thing. Lizzy is not a bit better than the others: and I am sure she is not half so handsome as Jane, nor half so good-humoured as Lydia. But you are always giving _her_ the preference.” “They have none of them much to recommend them,” replied he: “they are all silly and ignorant like other girls; but Lizzy has something more of quickness than her sisters.” “Mr. Bennet, how can you abuse your own children in such a way? You take delight in vexing me. You have no compassion o

## 9) RAG answering
The model is instructed to answer only from the provided context and cite sources.

In [10]:
SYSTEM_PROMPT = (
    "You are a helpful assistant. Answer ONLY and ONLY using the provided context. "
    "If the answer is not in the context, say: 'I don't know based on the provided context.' "
    "Cite sources in square brackets like [doc_id#chunk_index] for each key claim."
)

def answer_with_rag(question, k=None, temperature=0.2):
    hits = retrieve(question, k=k)
    context = build_context(hits)
    user_prompt = f"Question: {question}\n\nContext:\n{context}"
    answer = ollama_chat(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        temperature=temperature,
    )
    return {"question": question, "answer": answer, "hits": hits}

import textwrap

def show(out, ctx_preview_chars=400):
    print("[Q]", out["question"])
    print("\n[ANSWER]\n")
    print(textwrap.fill(out["answer"].strip(), width=100))
    print("\n[TOP HITS]")
    for i, h in enumerate(out["hits"], 1):
        print(f"  {i}. {h['title']}  [{h['doc_id']}#{h['chunk_index']}]  distance={h['distance']:.4f}")
    print("\n[CONTEXT PREVIEW]\n")
    print(build_context(out["hits"])[:ctx_preview_chars])

show(answer_with_rag("How does Victor Frankenstein create life?", k=5))

[Q] How does Victor Frankenstein create life?

[ANSWER]

According to the provided context, Victor Frankenstein creates life by infusing animation into an
inanimate body. He uses his knowledge of science and mechanics to collect and arrange materials, and
then applies a process that brings the being to life [frankenstein#56]. The exact details of this
process are not specified, but it is described as a "work of inconceivable difficulty and labour"
[frankenstein#56] and involves the creation of a being with a complex organization, including
muscles, veins, and other bodily systems.

[TOP HITS]
  1. Frankenstein  [frankenstein#62]  score=0.5906234979629517
  2. Frankenstein  [frankenstein#127]  score=0.6023390293121338
  3. Frankenstein  [frankenstein#56]  score=0.6087899208068848
  4. Frankenstein  [frankenstein#305]  score=0.6146427392959595
  5. Frankenstein  [frankenstein#57]  score=0.616858184337616

[CONTEXT PREVIEW]

[frankenstein#62] Frankenstein
it breathed hard, and a convulsiv

## 10) Maintenance helpers
Use these to wipe Chroma and/or artifacts if you want a clean rebuild.

In [11]:
def wipe_chroma():
    client = chroma_client()
    try:
        client.delete_collection(SETTINGS.CHROMA_COLLECTION)
        print(f"Deleted Chroma collection: {SETTINGS.CHROMA_COLLECTION}")
    except Exception as e:
        print(f"Could not delete collection: {e}")

def wipe_artifacts():
    ap = artifacts_paths()
    for key, p in ap.items():
        if key == "base":
            continue
        try:
            if p.exists():
                p.unlink()
                print(f"Deleted: {p}")
        except Exception as e:
            print(f"Could not delete {p}: {e}")

def info():
    m = load_manifest()
    return {
        "chroma_path": SETTINGS.CHROMA_PATH,
        "collection": SETTINGS.CHROMA_COLLECTION,
        "chroma_count": chroma_count(),
        "artifacts_dir": SETTINGS.ARTIFACTS_DIR,
        "has_manifest": bool(m),
        "expected_chunks": (m or {}).get("chunk_count"),
        "settings": settings_signature(),
    }

info()

# wipe_chroma()
# wipe_artifacts()


{'chroma_path': './chroma_db',
 'collection': 'rag_demo',
 'chroma_count': 8509,
 'artifacts_dir': 'rag_artifacts',
 'has_manifest': True,
 'expected_chunks': 8509,
 'settings': {'OLLAMA_URL': 'http://localhost:11434',
  'CHAT_MODEL': 'llama3.1:8b',
  'EMBED_MODEL': 'embeddinggemma:latest',
  'WORDS_PER_CHUNK': 300,
  'OVERLAP_WORDS': 60,
  'CHROMA_COLLECTION': 'rag_demo',
  'CHROMA_DISTANCE': 'cosine'}}

answer_with_rag(question)
    ↓
retrieve(question, k)
    ├─ ollama_embed([question]) → single embedding
    ├─ l2_normalize() → unit norm (cosine distance)
    ├─ coll.query(embeddings, n_results=k) → Chroma search
    └─ return hits [{ text, doc_id, chunk_index, title, source, score }, ...]
    ↓
build_context(hits)
    └─ format each hit as "[doc_id#chunk_index] title\ntext\n---\n"
    ↓
ollama_chat([system_prompt, user_prompt with context])
    └─ return answer (with citations)
